# The forge: concept views

Makes an asset's concept views from the prompts `tools/forge prompts <asset>` wrote, and records
what made each: the model, its version, the seed and the prompt's hash. The images are working
material: they go back into `.forge/<asset>/concept/` and never ship.

**Not yet run.** Written without a Colab session to try it in: check the model id against the
Gemini API's current list before the first run, and correct this notebook in the change that
first runs it.

Upload `.forge/<asset>/prompts.json` to `MyDrive/wrela-forge/<asset>/` first. Set `GOOGLE_API_KEY`
in Colab's secrets.

In [ ]:
!pip -q install google-genai pillow

In [ ]:
import hashlib, json, os
from google.colab import drive, userdata
from google import genai
from google.genai import types

ASSET = "table"          # the asset's name in .forge
MODEL = "imagen-4.0-generate-001"   # check against the API's list before running
SEED = 1234
drive.mount("/content/drive")
root = f"/content/drive/MyDrive/wrela-forge/{ASSET}"
prompts = json.load(open(f"{root}/prompts.json"))
client = genai.Client(api_key=userdata.get("GOOGLE_API_KEY"))

In [ ]:
os.makedirs(f"{root}/concept", exist_ok=True)
made = []
for k, p in enumerate(prompts):
    out = client.models.generate_images(
        model=MODEL,
        prompt=p["prompt"],
        config=types.GenerateImagesConfig(number_of_images=1, seed=SEED + k, add_watermark=False),
    )
    path = f"{root}/concept/{p['view']}.png"
    out.generated_images[0].image.save(path)
    made.append({"view": p["view"], "file": f"concept/{p['view']}.png", "model": MODEL,
                 "seed": SEED + k, "prompt_sha256": hashlib.sha256(p["prompt"].encode()).hexdigest()})
json.dump(made, open(f"{root}/concepts.json", "w"), indent=2)
made

Copy `concept/` and `concepts.json` back into `.forge/<asset>/`, and add `concepts.json`'s entries
to `forge.json`'s `concepts` so `tools/forge record` keeps them.